# 01 — Data Exploration & Analysis

Exploratory analysis of the arXiv dataset used for training the Research Intelligence Model.

**Sections:**
1. Dataset loading and overview
2. Category distribution
3. Abstract length distribution
4. Quality label analysis
5. Citation distribution
6. Text preprocessing effects

In [ ]:
import sys
sys.path.insert(0, '../..')   # repo root

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

In [ ]:
from ml.data.arxiv_loader import load_dataframe, _generate_synthetic_data, TARGET_CATEGORIES
from ml.data.preprocessor import preprocess_dataframe, _quality_heuristic

# Load or generate data
try:
    df = load_dataframe('../data/arxiv.csv')
except Exception:
    print('Generating synthetic data for demo...')
    df = _generate_synthetic_data(n=2000)

print(f'Dataset shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

## 1. Category Distribution

In [ ]:
cat_counts = df['category'].value_counts()

fig, ax = plt.subplots(figsize=(12, 5))
cat_counts.plot(kind='bar', ax=ax, color=sns.color_palette('muted', len(cat_counts)))
ax.set_title('Research Area Distribution (arXiv Categories)', fontsize=14)
ax.set_xlabel('Category')
ax.set_ylabel('Number of Papers')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../outputs/figures/category_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nClass imbalance ratio (max/min): {cat_counts.max() / cat_counts.min():.1f}x')
print(f'\nCategory counts:\n{cat_counts}')

## 2. Abstract Length Distribution

In [ ]:
df['abstract_len'] = df['abstract'].str.split().str.len()
df['title_len']    = df['title'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['abstract_len'], bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(df['abstract_len'].median(), color='red', linestyle='--',
                label=f"Median: {df['abstract_len'].median():.0f} words")
axes[0].set_title('Abstract Length Distribution')
axes[0].set_xlabel('Word Count')
axes[0].legend()

axes[1].hist(df['title_len'], bins=20, color='coral', edgecolor='white')
axes[1].axvline(df['title_len'].median(), color='navy', linestyle='--',
                label=f"Median: {df['title_len'].median():.0f} words")
axes[1].set_title('Title Length Distribution')
axes[1].set_xlabel('Word Count')
axes[1].legend()

plt.tight_layout()
plt.savefig('../outputs/figures/length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(df[['abstract_len', 'title_len']].describe())

## 3. Quality Label Analysis

In [ ]:
df['quality_label'] = df['abstract'].apply(lambda a: _quality_heuristic(str(a)))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Overall quality distribution
ql_counts = df['quality_label'].value_counts()
axes[0].pie(
    ql_counts.values,
    labels=['Poor', 'Well-structured'] if ql_counts.index[0] == 0 else ['Well-structured', 'Poor'],
    autopct='%1.1f%%',
    colors=['#e74c3c', '#2ecc71'],
    startangle=90,
)
axes[0].set_title('Abstract Quality Distribution')

# Quality by category
qual_by_cat = df.groupby('category')['quality_label'].mean().sort_values()
qual_by_cat.plot(kind='barh', ax=axes[1],
                  color=['#e74c3c' if v < 0.5 else '#2ecc71' for v in qual_by_cat])
axes[1].axvline(0.5, color='gray', linestyle='--')
axes[1].set_title('Mean Quality Score by Category')
axes[1].set_xlabel('Proportion Well-structured')

plt.tight_layout()
plt.savefig('../outputs/figures/quality_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Citation Score Distribution

In [ ]:
import math

if 'citations' in df.columns:
    df['log_citations'] = df['citations'].apply(lambda c: math.log1p(float(c)))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].hist(df['citations'].clip(upper=500), bins=50, color='purple', edgecolor='white')
    axes[0].set_title('Citation Count Distribution (clipped at 500)')
    axes[0].set_xlabel('Citations')

    axes[1].hist(df['log_citations'], bins=40, color='mediumpurple', edgecolor='white')
    axes[1].set_title('Log1p-transformed Citation Distribution')
    axes[1].set_xlabel('log1p(citations)')

    plt.tight_layout()
    plt.savefig('../outputs/figures/citation_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(df['citations'].describe())
else:
    print('No citation column in this dataset.')

## 5. Preprocessing Effects

In [ ]:
from ml.data.preprocessor import clean_text

examples = [
    'We minimise $L = \\sum_{i=1}^{n} \\|y_i - \\hat{y}_i\\|^2$ using gradient descent [Smith 2020].',
    'See https://arxiv.org/abs/1234 for details. Email: contact@lab.edu',
    'As shown in [1, 2, 3] and (Doe et al., 2019), our approach achieves SOTA.',
]

print('Preprocessing examples:\n')
for ex in examples:
    cleaned = clean_text(ex)
    print(f'  BEFORE: {ex}')
    print(f'  AFTER:  {cleaned}')
    print()

## Summary Statistics

In [ ]:
print('=== Dataset Summary ===')
print(f'Total samples:       {len(df):,}')
print(f'Unique categories:   {df["category"].nunique()}')
print(f'Avg abstract length: {df["abstract_len"].mean():.0f} words')
print(f'Quality rate:        {df["quality_label"].mean():.1%}')
if 'citations' in df.columns:
    print(f'Median citations:    {df["citations"].median():.0f}')
print()
print('Ready for training pipeline!')